# Mean Surface Clusters
The idea here is to take some mean of surface values (or possibly subset of) to cluster on -- the hope is that this can give insights into water masses on the shelf as well as in the basins (which is all the current method is realistically doing)

In [ ]:
from importlib import reload

import pandas as pd
import xarray as xr
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import gsw_xarray as gsw

import toolbox as tbx
reload(tbx)

import visualisation as viz
reload(viz)

RANDOM_SEED = 5

In [ ]:
ds = tbx.load_dataset()
interpolated_data = tbx.interpolate_dataset(ds)
regional_data = interpolated_data #tbx.drop_region(interpolated_data, region_choice='Chukchi Sea')
regional_data

In [ ]:
strait_data = tbx.select_region(
    interpolated_data,
    longitude_range=(186, 193), latitude_range=(65, 66.5)
)

tbx.plot_all_vals(strait_data)

In [ ]:
from argopy import DataFetcher

bottom_right = (50, -130)
top_left = (80, -180) # (74, 170.4)

argo_data_E = DataFetcher().region([top_left[1], bottom_right[1], bottom_right[0], top_left[0], 0, 10]).to_xarray()

bottom_right = (50, 180)
top_left = (80, 160)

argo_data_W = DataFetcher().region([top_left[1], bottom_right[1], bottom_right[0], top_left[0], 0, 10]).to_xarray()

In [ ]:
reload(tbx)

fig, ax = tbx.geoaxes(figsize=(12, 8))

tbx.plot_all_vals(
    ds, color=ds.BOTTOM_DEPTH, vmin=0, vmax=150, subplots = (fig, ax), 
    pt_size = 5, alpha=1,
    cbar_kwargs={'extend':'max', 'pad':0.1, 'label': 'Bottom Depth (m)', 'shrink': 0.7}
)

ARGO_color = 'gray'

tbx.plot_all_vals(
    argo_data_E, color=ARGO_color, subplots = (fig, ax), zorder=-1
)

tbx.plot_all_vals(
    argo_data_W, color=ARGO_color, subplots = (fig, ax), zorder=-1
)

ax.set_extent([
    160, -130,
    45, 75
])
ax.gridlines(draw_labels=True)

fig.tight_layout()

## Surface Mean Density

In [ ]:
regional_data['Absolute_Salinity'] = gsw.conversions.SA_from_SP(
    regional_data.Merged_Salinity, 
    regional_data.PRESSURE,
    regional_data.LONGITUDE, regional_data.LATITUDE
)

## NOTE THIS IS POTENTIAL TEMPERATURE I JUST MESSED UP AND DIDN'T HAVE TIME TO CHANGE ALL TEH VARIABLES
regional_data['Cons_Temperature'] = gsw.conversions.pt_from_t(
    regional_data.Absolute_Salinity, 
    regional_data.Temperature, 
    regional_data.PRESSURE,
    p_ref=0
)

regional_data['Sigma0'] = tbx.calculate_sigma_t(regional_data, as_var='Absolute_Salinity', ct_var='Cons_Temperature')
surface_mean = regional_data.sel(PRESSURE=slice(10, 50)).mean(dim='PRESSURE')

sig_mask = ~np.isnan(surface_mean['Sigma0'])

clean_means = surface_mean.where(sig_mask, drop=True)


In [ ]:
full_ds = regional_data.where(sig_mask, drop=True)

In [ ]:
tbx.plot_all_vals(clean_means, color=clean_means['Cons_Temperature'])

In [ ]:
strait_ds = tbx.select_region(
    clean_means,
    longitude_range=(186, 193), latitude_range=(65, 66.5)
)

strait_sal = strait_ds['Absolute_Salinity'].data.reshape(-1, 1)
strait_temp = strait_ds['Cons_Temperature'].data.reshape(-1, 1)
strait_density = strait_ds['Sigma0'].data.reshape(-1, 1)

training_data = np.concatenate((strait_sal, strait_temp, strait_density), axis=1)
training_data = tbx.normalise_array(training_data)

In [ ]:
final_sal = clean_means['Absolute_Salinity'].data.reshape(-1, 1)
final_temp = clean_means['Cons_Temperature'].data.reshape(-1, 1)
final_density = clean_means['Sigma0'].data.reshape(-1, 1)

full_data = np.concatenate((final_sal, final_temp, final_density), axis=1)
full_data = tbx.normalise_array(full_data)

In [ ]:
full_data.shape

# The rest of the code

In [ ]:
reload(tbx)

from tqdm import tqdm

attempts = 50

subset_size = training_data.shape[0] // 3

AS = []
BS = []

max_clusters = 20
for n_clusters in tqdm(range(2, max_clusters)):
    temp_AS = []
    temp_BS = []
    for _ in range(attempts):
        subsampled_data = tbx.random_subset(training_data, subset_size)
        
        gmm = GaussianMixture(n_components=n_clusters, random_state=RANDOM_SEED, reg_covar=1e-5)
        gmm.fit(subsampled_data)
    
        temp_AS.append(gmm.aic(subsampled_data))
        temp_BS.append(gmm.bic(subsampled_data))

    AS.append(temp_AS)
    BS.append(temp_BS)

AS = np.array(AS)
BS = np.array(BS)

In [ ]:
reload(tbx)

fig, ax = plt.subplots(figsize=(8, 5))

tbx.plot_stats(
    AS, ax, label='AIC', 
    color='black',
)

tbx.plot_stats(
    BS, ax, label='BIC',
    color='blue',
)

ax.set_ylabel('Score')
ax.set_xlabel('Number of Components')

ax.legend()

'yo'

In [ ]:
from sklearn.mixture import GaussianMixture
from tqdm import tqdm

AS = []
BS = []

max_clusters = 20
for n_clusters in tqdm(range(2, max_clusters)):
    gmm = GaussianMixture(n_components=n_clusters, random_state=RANDOM_SEED, reg_covar=5e-6).fit(training_data)

    AS.append(gmm.aic(training_data))
    BS.append(gmm.bic(training_data))

plt.plot(range(2, max_clusters), AS, label='AS')
plt.plot(range(2, max_clusters), BS, label='BS')
plt.xticks(range(2, max_clusters))
plt.legend()

In [ ]:
model = GaussianMixture(n_components=6, random_state=RANDOM_SEED)
model.fit(full_data)

clusters = model.predict(full_data)

In [ ]:
## Color mapping functions

def dec_to_hex(n):
    '''
    for use in rgba_to_hex

    scales numbers from 0 to 1 -> hex from 1 -> 225
    '''
    
    decimal_value = round(n * 256)
    if decimal_value == 256:
        decimal_value = 255
    hex_val = f'{decimal_value:02x}'

    return hex_val

def rgba_to_hex(arr):
    '''
    assuming input is in form from colormap
    (r, g, b, a)
    '''

    rgb_values = arr[:3]

    return '#' + ''.join([dec_to_hex(num) for num in rgb_values])

cmap = mpl.colormaps['viridis']
_cid = np.linspace(0, 1, len(set(clusters)))

colors = [
    rgba_to_hex(cmap(i)) for i in _cid
]

colors

In [ ]:
reload(tbx)
num_clusters = len(set(clusters))

fig, axs = tbx.geoaxes(2, 3, figsize=(8, 8))

@tbx.cluster_wrapper(clean_means, clusters)
def plot_cluster(cid, cds):
    
    ax = axs[cid]
        
    ax.set_title(f'Cluster {cid}')
    im = tbx.plot_all_vals(cds, ax=ax, color=colors[cid], alpha=0.6)

    return im

fig.tight_layout()

plot_cluster()

In [ ]:
reload(tbx)
num_clusters = len(set(clusters))

fig, axs = tbx.geoaxes(1, 4, figsize=(10, 5))
FIG, AXS = tbx.geoaxes(1, 3, figsize=(8, 5))

@tbx.cluster_wrapper(clean_means, clusters)
def plot_cluster(cid, cds):
    
    if cid < 4:
        ax = axs[cid]

    else:
        ax = AXS[cid-4]
        
    ax.set_title(f'Cluster {cid}')
    im = tbx.plot_all_vals(cds, ax=ax, color=colors[cid], alpha=0.6)

    return im

plot_cluster()

In [ ]:
@tbx.cluster_wrapper(clean_means, clusters)
def plot_cluster(cid, cds):
    fig, ax = tbx.geoaxes()

    ax.set_title(f'Cluster {cid}')
    im = tbx.plot_all_vals(cds, ax=ax, color=colors[cid], alpha=0.6)

    return im

plot_cluster()

In [ ]:
clean_means

In [ ]:


fig, ax = tbx.geoaxes(figsize=(8, 4))

deep_mask = clean_means.BOTTOM_DEPTH >= 300

deep_ds = clean_means.where(deep_mask, drop=True)
deep_clusters = clusters[deep_mask]

@tbx.cluster_wrapper(deep_ds, deep_clusters)
def plot_cluster(cid, cds):

    im = tbx.plot_all_vals(cds, ax=ax, color=colors[cid], alpha=0.6, label=f'Cluster {cid}')

    return im

plot_cluster()
leg = ax.legend(markerscale=4)

for lh in leg.legend_handles:
    lh.set_alpha(1)

In [ ]:


fig, ax = tbx.geoaxes(figsize=(8, 4))

@tbx.cluster_wrapper(clean_means, clusters)
def plot_cluster(cid, cds):

    im = tbx.plot_all_vals(cds, ax=ax, color=colors[cid], alpha=0.6, label=f'Cluster {cid}')

    return im

plot_cluster()
leg = ax.legend(markerscale=4)

for lh in leg.legend_handles:
    lh.set_alpha(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

n_pts = 80
pt_size = 2

full_AS = clean_means.Absolute_Salinity
full_PT = clean_means.Cons_Temperature

AS_min = min(float(full_AS.min().data), 27)
AS_max = max(float(full_AS.max().data), 36)
AS_range = np.linspace(AS_min, AS_max, n_pts)

PT_min = min(float(full_PT.min().data), -2)
PT_max = max(float(full_PT.max().data)+(pt_size/25), 12)
PT_range = np.linspace(PT_min, PT_max, n_pts)

xticks = np.arange(np.floor(AS_min), np.ceil(AS_max)+1, 1)
yticks = np.arange(np.floor(PT_min), np.ceil(PT_max)+1, 1)

AS_grid = AS_range.reshape(-1, 1).repeat(n_pts, axis=1).T
PT_grid = PT_range.reshape(-1, 1).repeat(n_pts, axis=1)

sigma0 = gsw.density.sigma0(AS_grid, PT_grid)+1000

clevels = np.arange(np.floor(sigma0.min()), np.ceil(sigma0.max())+1, 1)

ax.set_xticks(xticks)
ax.set_yticks(yticks)

CONTOURS = ax.contour(AS_range, PT_range, sigma0, colors='lightgray', levels=clevels, alpha=0.5, zorder=-1)
ax.clabel(CONTOURS, CONTOURS.levels)


@tbx.cluster_wrapper(clean_means, clusters)
def basic_TS_plot(cds, cid, ax=ax):

    print(f'Plotting cluster {cid+1}/{len(set(clusters))}')

    color = colors[cid]

    Ctemp = cds.Cons_Temperature
    Csal = cds.Absolute_Salinity

    im = ax.scatter(Csal, Ctemp, s=1, color=color, label=f'Cluster {cid}')

    return im

basic_TS_plot()
ax.legend(markerscale=4, loc='lower left')

ax.set_xlabel('Absolute Salinity')
ax.set_ylabel('Potential Temperature ($^\\circ$C)')

ax.set_xlim(20)

In [ ]:
import seaborn as sns
import pandas as pd

